# 실관람객 리뷰

In [ ]:
# from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

In [29]:
"""
NAVER ‘네티즌 관람평’ 크롤러  —  URL( naver_search_url ) → 스포일러 포함 → 리뷰 전량 수집
────────────────────────────────────────────────────────────────────────
입력  : final_with_url.xlsx  (필수 컬럼: 영화명, naver_search_url)
출력  : naver_reviews_full.xlsx  (리뷰·집계 정보 누적 저장)
"""

import time, random, re
from pathlib import Path
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import (
    NoSuchElementException, TimeoutException, ElementClickInterceptedException
)
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

# ───────── 사용자 설정 ──────────────────────────
HEADLESS   = False                    # True → 창 숨김, False → 창 표시
START_ROW  = 0                       # URL 리스트 시작 위치(0-base)
SAVE_FILE  = Path("naver_reviews_full.xlsx")
SCROLL_INT = (500, 800)              # panel.scrollBy 픽셀(Random)
SLEEP_INT  = (0.25, 0.35)            # 스크롤 딜레이(Random)
UA_STRING  = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
              "AppleWebKit/537.36 (KHTML, like Gecko) "
              "Chrome/137.0.0.0 Safari/537.36")
# ───────────────────────────────────────────────


# 1) URL 리스트 로드 ─────────────────────────────
df_url = pd.read_excel("final_with_url.xlsx")
url_list = df_url.loc[START_ROW:, ["영화명", "naver_search_url"]].values.tolist()

# 2) Selenium 드라이버 준비 ─────────────────────
opt = Options()
if HEADLESS:
    opt.add_argument("--headless=new")
opt.add_argument(f"--user-agent={UA_STRING}")
opt.add_argument("--disable-gpu")
opt.add_argument("--blink-settings=imagesEnabled=false")
opt.page_load_strategy = "eager"      # 렌더링 최소화
driver = webdriver.Chrome(options=opt)


# ───────── 헬퍼 함수들 ──────────────────────────
def txt(xp: str, root=driver):
    """XPath 텍스트 추출 (없으면 None)"""
    try:
        return root.find_element(By.XPATH, xp).text.strip()
    except NoSuchElementException:
        return None


def clean(s):
    """엑셀 금지 제어문자 제거"""
    return ILLEGAL_CHARACTERS_RE.sub("", s) if isinstance(s, str) else s


def save_rows(rows):
    """리스트[dict] → 엑셀 append 저장"""
    if not rows:
        return
    df_new = pd.DataFrame(rows).applymap(clean)
    if SAVE_FILE.exists():
        with pd.ExcelWriter(SAVE_FILE, mode="a",
                            if_sheet_exists="overlay",
                            engine="openpyxl") as w:
            start = w.sheets["Sheet1"].max_row
            df_new.to_excel(w, index=False, header=False, startrow=start)
    else:
        df_new.to_excel(SAVE_FILE, index=False)


def click_spoiler(driver, timeout=5) -> bool:
    """
    절대 XPath로 [스포일러 포함] 버튼을 찾아 클릭.
    성공 → True, 버튼 없음/실패 → False
    """
    ABS_XP = ("/html/body/div[3]/div[2]/div[1]/div[1]/div[3]/div[2]/div/div/"
              "div[4]/div[2]/div[2]/button/span[1]/span[1]")  # ← 절대 XPath

    try:
        btn = WebDriverWait(driver, timeout).until(
            EC.element_to_be_clickable((By.XPATH, ABS_XP))
        )
        driver.execute_script(
            "arguments[0].scrollIntoView({block:'center', inline:'center'});", btn)
        driver.execute_script("arguments[0].click();", btn)
        time.sleep(0.3)
        return True
    except (TimeoutException, ElementClickInterceptedException):
        return False


# 3) 메인 크롤 루프 ─────────────────────────────
try:
    for title, url in url_list:
        print(f"\n▶ {title}")
        driver.get(url)
        time.sleep(1)                # 첫 로딩 대기

        # (1) 스포일러 버튼 클릭 시도
        click_spoiler(driver)

        # (2) 리뷰 패널 찾기
        try:
            panel = driver.find_element(
                By.XPATH, '//*[@id="main_pack"]//div[4]/div[4]')
        except NoSuchElementException:
            print("   ↳ 리뷰 모듈 없음 – skip")
            continue

        # (3) 패널 내부 스크롤 (끝까지)
        prev_cnt = same_cnt = 0
        MAX_SAME = 4
        while True:
            driver.execute_script(
                "arguments[0].scrollBy(0, arguments[1]);",
                panel, random.randint(*SCROLL_INT))
            time.sleep(random.uniform(*SLEEP_INT))

            li_cnt = len(panel.find_elements(By.XPATH, ".//ul/li"))
            if li_cnt > prev_cnt:
                prev_cnt, same_cnt = li_cnt, 0
                continue

            same_cnt += 1
            if same_cnt < MAX_SAME:
                continue

            top  = driver.execute_script("return arguments[0].scrollTop;", panel)
            maxT = driver.execute_script(
                     "return arguments[0].scrollHeight - arguments[0].clientHeight;", panel)
            if top >= maxT - 20:
                break
            same_cnt = 0

        if prev_cnt == 0:
            print("   ↳ 리뷰 0개 – skip")
            continue
        print(f"   ↳ 리뷰 {prev_cnt:,}개 로드 완료")

        # (4) 집계 info
        aud = txt('//ul/li[1]/div/div[1]/span[1]')
        ms  = txt('//*[@id="main_pack"]//ul/li[1]/div/div[2]/div[1]/span[2]/span')
        fs  = txt('//*[@id="main_pack"]//ul/li[1]/div/div[2]/div[2]/span[2]/span')
        mr  = txt('//ul/li[3]//li[1]/span/em')
        fr  = txt('//ul/li[3]//li[2]/span/em')
        rct = txt('//*[@id="main_pack"]//ul/li[1]/div/div[1]/span[3]')

        # (5) 리뷰 파싱
        rows = []
        for li in panel.find_elements(By.XPATH, ".//ul/li"):
            try:
                more = li.find_element(By.XPATH, "./div[2]/div/button")
                if more.is_displayed():
                    driver.execute_script("arguments[0].click();", more)
                    time.sleep(0.05)
            except NoSuchElementException:
                pass

            review = txt("./div[2]/div/span[2]", li)
            if not review:
                continue
            raw_sc = txt("./div[1]/div/div[2]", li)
            score  = re.search(r"(\d{1,2})$", raw_sc).group(1) if raw_sc else None

            rows.append({
                "영화명":         clean(title),
                "작성일":         clean(txt("./dl/dd[2]", li)),
                "작성자":         clean(txt("./dl/dd[1]", li)),
                "감상평":         clean(review),
                "별점":           score,
                "공감수":        txt("./div[3]/button[1]/span", li),
                "비공감수":       txt("./div[3]/button[2]/span", li),
                "실관람객 평점":  aud,
                "남자 평점":      ms,
                "여자 평점":      fs,
                "남자 성비":     clean(f"{mr}%" if mr else None),
                "여자 성비":     clean(f"{fr}%" if fr else None),
                "리뷰 참여 명수": rct
            })

        save_rows(rows)
        print(f"   ↳ {len(rows)}행 저장 → {SAVE_FILE.name}")

except KeyboardInterrupt:
    print("\n⏹️  사용자 중단 – 이미 저장된 행은 보존")

finally:
    driver.quit()
    if SAVE_FILE.exists() and SAVE_FILE.stat().st_size:
        print(f"\n✅ 최종 저장 위치: {SAVE_FILE.resolve()}")
    else:
        print("\n⚠️ 결과가 비어 있습니다.")



▶ 남산의 부장들

✅ 최종 저장 위치: C:\Users\82104\anaconda_projects_files(2025)\hanyang_paper(RE)\naver_reviews_full.xlsx


TypeError: object of type 'NoneType' has no len()

# ＃ 네티즌 관람평 크롤링

In [15]:
"""
NAVER ‘네티즌 관람평’ 크롤러  –  절대 XPath + 점수별 비율 % 추출
────────────────────────────────────────────────────────────────────────
입력 : final_with_url.xlsx    (컬럼: 영화명, naver_search_url)
출력 : naver_reviews_ott.xlsx (리뷰·집계·점수비율 정보 누적 저장)
"""

import time, random, re
from pathlib import Path
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import (
    NoSuchElementException, TimeoutException, ElementClickInterceptedException
)
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

# ═════════════ 사용자 설정 ═════════════
HEADLESS   = True
SAVE_FILE  = Path("naver_reviews_ott.xlsx")
UA_STRING  = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
              "AppleWebKit/537.36 (KHTML, like Gecko) "
              "Chrome/137.0.0.0 Safari/537.36")
SCROLL_INT = (500, 800)
SLEEP_INT  = (0.25, 0.35)
# ═════════════════════════════════════

# ── 절대 XPath 상수 ─────────────────────────────────
XP_TAB_REVIEW  = ("/html/body/div[3]/div[2]/div[1]/div[1]/div[3]/div[2]/div/"
                  "div/div[1]/div/div/ul/li[2]/a/span")
XP_BTN_SPOILER = ("/html/body/div[3]/div[2]/div[1]/div[1]/div[3]/div[2]/div/"
                  "div/div[5]/div[2]/div[2]/button/span[1]/span[1]")
XP_PANEL       = ("/html/body/div[3]/div[2]/div[1]/div[1]/div[3]/div[2]/div/"
                  "div/div[5]/div[4]")

# 상단 집계
XP_ACTUAL = ("/html/body/div[3]/div[2]/div[1]/div[1]/div[3]/div[2]/div/"
             "div/div[3]/div[1]/div/div/ul/li[1]/div/div[1]/span[1]")
XP_JOIN   = ("/html/body/div[3]/div[2]/div[1]/div[1]/div[3]/div[2]/div/"
             "div/div[3]/div[1]/div/div/ul/li[1]/div/div[1]/span[3]")
XP_M_SCORE = ("/html/body/div[3]/div[2]/div[1]/div[1]/div[3]/div[2]/div/"
              "div/div[3]/div[1]/div/div/ul/li[1]/div/div[2]/div[1]/span[2]/span")
XP_F_SCORE = ("/html/body/div[3]/div[2]/div[1]/div[1]/div[3]/div[2]/div/"
              "div/div[3]/div[1]/div/div/ul/li[1]/div/div[2]/div[2]/span[2]/span")
XP_M_RATIO = ("/html/body/div[3]/div[2]/div[1]/div[1]/div[3]/div[2]/div/"
              "div/div[3]/div[1]/div/div/ul/li[3]/div/div/ul/li[1]/span/em")
XP_F_RATIO = ("/html/body/div[3]/div[2]/div[1]/div[1]/div[3]/div[2]/div/"
              "div/div[3]/div[1]/div/div/ul/li[3]/div/div/ul/li[2]/span/em")

# 점수별 비율 %
XP_RATE_9  = ("/html/body/div[3]/div[2]/div[1]/div[1]/div[3]/div[2]/div/"
              "div/div[3]/div[1]/div/div/ul/li[2]/div/ul/li[1]/div[2]")
XP_RATE_7  = ("/html/body/div[3]/div[2]/div[1]/div[1]/div[3]/div[2]/div/"
              "div/div[3]/div[1]/div/div/ul/li[2]/div/ul/li[2]/div[2]")
XP_RATE_5  = ("/html/body/div[3]/div[2]/div[1]/div[1]/div[3]/div[2]/div/"
              "div/div[3]/div[1]/div/div/ul/li[2]/div/ul/li[3]/div[2]")
XP_RATE_3  = ("/html/body/div[3]/div[2]/div[1]/div[1]/div[3]/div[2]/div/"
              "div/div[3]/div[1]/div/div/ul/li[2]/div/ul/li[4]/div[2]")
XP_RATE_1  = ("/html/body/div[3]/div[2]/div[1]/div[1]/div[3]/div[2]/div/"
              "div/div[3]/div[1]/div/div/ul/li[2]/div/ul/li[5]/div[2]")
# ──────────────────────────────────────

def clean(s):
    return ILLEGAL_CHARACTERS_RE.sub("", s) if isinstance(s, str) else s

def click_xpath(driver, xp, timeout=8):
    try:
        el = WebDriverWait(driver, timeout).until(
            EC.element_to_be_clickable((By.XPATH, xp)))
        driver.execute_script(
            "arguments[0].scrollIntoView({block:'center'});", el)
        driver.execute_script("arguments[0].click();", el)
        time.sleep(0.4)
        return True
    except (TimeoutException, ElementClickInterceptedException):
        return False

def grab_text(xp):
    try:
        return driver.find_element(By.XPATH, xp).text.strip()
    except NoSuchElementException:
        return None

def save_append(rows):
    if not rows: return
    df_new = pd.DataFrame(rows).applymap(clean)
    if SAVE_FILE.exists():
        with pd.ExcelWriter(SAVE_FILE, mode="a",
                            if_sheet_exists="overlay",
                            engine="openpyxl") as w:
            start = w.sheets["Sheet1"].max_row
            df_new.to_excel(w, index=False, header=False, startrow=start)
    else:
        df_new.to_excel(SAVE_FILE, index=False)

# ── 드라이버 ─────────────────────────────
opt = Options()
if HEADLESS: opt.add_argument("--headless=new")
opt.add_argument(f"--user-agent={UA_STRING}")
opt.add_argument("--disable-gpu")
opt.page_load_strategy = "eager"
driver = webdriver.Chrome(options=opt)
WAIT   = WebDriverWait(driver, 10)

# ── URL 리스트 ───────────────────────────
todo = (pd.read_excel("final_with_url.xlsx")
          [["영화명", "naver_search_url"]].values.tolist())

try:
    for title, url in todo:
        print(f"\n▶ {title}")
        driver.get(url)

        if not click_xpath(driver, XP_TAB_REVIEW):
            print("   ↳ 탭 없음 – skip"); continue
        click_xpath(driver, XP_BTN_SPOILER, timeout=5)

        try:
            panel = WAIT.until(EC.presence_of_element_located((By.XPATH, XP_PANEL)))
        except TimeoutException:
            print("   ↳ 패널 못 찾음 – skip"); continue

        try:
            WebDriverWait(driver, 15).until(
                lambda d: panel.find_elements(By.CSS_SELECTOR, "ul > li"))
        except TimeoutException:
            print("   ↳ 리뷰 0 – skip"); continue

        # 스크롤 끝
        prev = same = 0
        while True:
            driver.execute_script("arguments[0].scrollBy(0, arguments[1]);",
                                  panel, random.randint(*SCROLL_INT))
            time.sleep(random.uniform(*SLEEP_INT))
            cur = len(panel.find_elements(By.CSS_SELECTOR, "ul > li"))
            if cur > prev: prev, same = cur, 0; continue
            same += 1
            if same < 4: continue
            if driver.execute_script(
                 "return arguments[0].scrollTop + arguments[0].clientHeight;", panel) \
               >= driver.execute_script(
                 "return arguments[0].scrollHeight;", panel) - 20:
                break; same = 0
        print(f"   ↳ 리뷰 {prev:,}")

        # 상단 집계 & 비율
        aud, join = grab_text(XP_ACTUAL), grab_text(XP_JOIN)
        ms,  fs   = grab_text(XP_M_SCORE), grab_text(XP_F_SCORE)
        mr,  fr   = grab_text(XP_M_RATIO), grab_text(XP_F_RATIO)
        rate9     = grab_text(XP_RATE_9)
        rate7     = grab_text(XP_RATE_7)
        rate5     = grab_text(XP_RATE_5)
        rate3     = grab_text(XP_RATE_3)
        rate1     = grab_text(XP_RATE_1)

        # 리뷰 파싱
        rows = []
        for li in panel.find_elements(By.CSS_SELECTOR, "ul > li"):
            review = li.get_attribute("data-report-title").strip()
            writer = li.get_attribute("data-report-writer-id")
            date   = li.get_attribute("data-report-time")[:10].replace(" ","")
            raw    = li.find_element(By.XPATH, "./div[1]/div/div[2]") \
                        .get_attribute("textContent").strip()
            score  = re.search(r"(\d+)\s*$", raw).group(1)
            like   = li.find_element(By.XPATH, "./div[3]/button[1]/span").text
            hate   = li.find_element(By.XPATH, "./div[3]/button[2]/span").text

            rows.append({
                "영화명": title, "작성일": date, "작성자": writer,
                "감상평": review, "별점": score,
                "공감수": like, "비공감수": hate,
                "실관람객 평점": aud, "리뷰 참여 명수": join,
                "남자 평점": ms, "여자 평점": fs,
                "남자 성비": mr, "여자 성비": fr,
                "9-10 비율": rate9, "7-8 비율": rate7,
                "5-6 비율": rate5, "3-4 비율": rate3, "1-2 비율": rate1
            })

        save_append(rows)
        print(f"   ↳ {len(rows)}행 저장")

finally:
    driver.quit()
    if SAVE_FILE.exists():
        print(f"\n✅ 완료 – {SAVE_FILE.resolve()}")
    else:
        print("\n⚠️ 결과가 비어 있습니다.")



▶ 남산의 부장들
   ↳ 리뷰 300


C:\Users\82104\AppData\Local\Temp\ipykernel_29888\3768030676.py:89: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_new = pd.DataFrame(rows).applymap(clean)


   ↳ 300행 저장

▶ 다만 악에서 구하소서
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 히트맨
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 테넷
   ↳ 리뷰 300
   ↳ 300행 저장

▶ #살아있다
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 강철비2: 정상회담
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 삼진그룹 영어토익반
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 도굴
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 클로젯
   ↳ 탭 없음 – skip

▶ 해치지않아
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 결백
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 국제수사
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 침입자
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 이웃사촌
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 소리도 없이
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 원더 우먼 1984
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 런
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 수퍼 소닉
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 애프터: 그 후
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 서치 아웃
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 테슬라
   ↳ 리뷰 136
   ↳ 136행 저장

▶ 추억의 검정고무신
   ↳ 리뷰 0 – skip

▶ 밥정
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 미드나이트 스카이
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 헌트
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 트라이얼 오브 더 시카고 7
   ↳ 리뷰 195
   ↳ 195행 저장

▶ 맹크
   ↳ 탭 없음 – skip

▶ 힐빌리의 노래
   ↳ 리뷰 284
   ↳ 284행 저장

▶ 런 보이 런
   ↳ 리뷰 105
   ↳ 105행 저장

▶ 모가디슈
   ↳ 리뷰 300
   ↳ 300행 저장

▶ 싱크홀
   ↳ 